# Ferramenta de navegador com visualização ao vivo usando Amazon Nova Act SDK

## Visão Geral

Neste tutorial, aprenderemos como usar o SDK Nova Act para interagir com a ferramenta Browser do Amazon Bedrock Agentcore e visualizar o navegador ao vivo.


### Detalhes do Tutorial


| Informação          | Detalhes                                                                          |
|:--------------------|:----------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional                                                                    |
| Tipo de agente      | Único                                                                             |
| Framework agêntico  | Nova Act                                                                          |
| Modelo LLM          | Modelo Amazon Nova Act                                                            |
| Componentes         | Usando NovaAct para interagir com a ferramenta de navegador ao vivo              |
| Vertical do tutorial| Cross-vertical                                                                    |
| Complexidade        | Fácil                                                                             |
| SDK utilizado       | Amazon BedrockAgentCore Python SDK, Nova Act                                      |

### Arquitetura do Tutorial

Neste tutorial, descreveremos como usar o Nova Act com a ferramenta de navegador e visualizá-lo ao vivo.  

Em nosso exemplo, enviaremos instruções em linguagem natural para o agente Nova Act executar tarefas no navegador Bedrock Agentcore e visualizar o navegador ao vivo.

<div style="text-align:left">
    <img src="./images/browser-tool.png" width="50%"/>
</div>

### Principais Recursos do Tutorial

* Usando a ferramenta de navegador e visualizando-a ao vivo
* Usando Nova Act com a ferramenta de navegador

## Pré-requisitos

Para executar este tutorial, você precisará de:
* Python 3.10+
* Credenciais AWS. Sua função/usuário IAM deve ter estas permissões https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/browser-onboarding.html#browser-credentials-config
* Amazon Bedrock AgentCore SDK
* Nova Act SDK e chave de API

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

## Usando NovaAct com a ferramenta Browser do Bedrock Agentcore com visualização ao vivo

Aqui, usaremos uma função auxiliar para conectar via Amazon DCV SDK à ferramenta de navegador do Bedrock Agentcore.




In [ ]:
%%writefile live_view_with_nova_act.py
from bedrock_agentcore.tools.browser_client import browser_session
from nova_act import NovaAct
from rich.console import Console
from rich.panel import Panel
import sys
import json
import argparse
sys.path.append("../interactive_tools")
from browser_viewer import BrowserViewerServer

console = Console()

from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print("usando região", region)

def live_view_with_nova_act(prompt, starting_page, nova_act_key, region="us-west-2"):
    """Executa o visualizador de navegador ao vivo com dimensionamento de tela."""
    console.print(
        Panel(
            "[bold cyan]Visualizador de Navegador ao Vivo[/bold cyan]\n\n"
            "Isso demonstra:\n"
            "• Visualização de navegador ao vivo com DCV\n"
            "• Tamanhos de tela configuráveis (não limitado a 900×800)\n"
            "• Callbacks de layout de tela adequados\n\n"
            "[yellow]Nota: Requer arquivos SDK do Amazon DCV[/yellow]",
            title="Visualizador de Navegador ao Vivo",
            border_style="blue",
        )
    )

    try:
        # Passo 1: Criar sessão de navegador
        with browser_session(region) as client:
            ws_url, headers = client.generate_ws_headers()

            # Passo 2: Iniciar servidor visualizador
            console.print("\n[cyan]Passo 3: Iniciando servidor visualizador...[/cyan]")
            viewer = BrowserViewerServer(client, port=8000)
            viewer_url = viewer.start(open_browser=True)

            # Passo 3: Mostrar recursos
            console.print("\n[bold green]Recursos do Visualizador:[/bold green]")
            console.print(
                "• Tela padrão: 1600×900 (configurado via callback displayLayout)"
            )
            console.print("• Opções de tamanho: 720p, 900p, 1080p, 1440p")
            console.print("• Atualizações de tela em tempo real")
            console.print("• Funcionalidade de assumir/liberar controle")

            console.print("\n[yellow]Pressione Ctrl+C para parar[/yellow]")

            # Passo 4: Usar Nova Act para interagir com o navegador
            with NovaAct(
                cdp_endpoint_url=ws_url,
                cdp_headers=headers,
                preview={"playwright_actuation": True},
                nova_act_api_key=nova_act_key,
                starting_page=starting_page,
            ) as nova_act:
                result = nova_act.act(prompt)
                console.print(f"\n[bold green]Resultado Nova Act:[/bold green] {result}")
        
    except Exception as e:
        console.print(f"\n[red]Erro: {e}[/red]")
        import traceback
        traceback.print_exc()
    finally:
        console.print("\n\n[yellow]Encerrando...[/yellow]")
        if "client" in locals():
            client.stop()
            console.print("✅ Sessão de navegador terminada")
    return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", required=True, help="Instrução de busca no navegador")
    parser.add_argument("--starting-page", required=True, help="URL inicial")
    parser.add_argument("--nova-act-key", required=True, help="Chave de API Nova Act")
    parser.add_argument("--region", default="us-west-2", help="Região AWS")
    args = parser.parse_args()

    result = live_view_with_nova_act(
        args.prompt, args.starting_page, args.nova_act_key, args.region
    )

    with open('result.txt', 'w') as f:
        f.write(str(result))

    console.print(f"\n[bold green]Resultado Nova Act:[/bold green] {result}")

#### Executando o script
Cole sua chave de API Nova Act abaixo antes de executar o script.

In [ ]:
NOVA_ACT_KEY= ''  ### Cole sua chave Nova Act aqui

In [ ]:
!python live_view_with_nova_act.py --prompt "Procure por macbooks e extraia os detalhes do primeiro" --starting-page "https://www.amazon.com/" --nova-act-key {NOVA_ACT_KEY}

### O que aconteceu nos bastidores? 
* Você instanciou um cliente Browser e iniciou uma sessão
* Em seguida, você usou o `BrowserViewerServer` para conectar à sessão do navegador e visualizar a sessão localmente
* Depois, você criou um Agente Nova Act e passou os detalhes da sessão do navegador para ele
* Você então enviou instruções em linguagem natural para o agente Nova Act e viu as ações ao vivo

## Lidando com CAPTCHAs no navegador 
A seguir, vamos ver como podemos lidar com captchas no navegador. O propósito dos captchas é garantir que um humano está interagindo com o site e não um bot. Então, não permitiremos que o agente resolva o captcha; em vez disso, assumiremos o controle - resolveremos o captcha e deixaremos o agente continuar.

Vamos criar nosso novo script. O Nova Act nos permite verificar se há um captcha na página - usaremos esse recurso e lidaremos com o captcha antes de deixar o Nova Act continuar. 

#### Quando o script começar a ser executado, você verá uma visualização local do navegador. Se você vir um captcha enquanto o script estiver em execução, resolva manualmente o captcha - o script aguarda até que o captcha seja resolvido. 

##### Nota: Se você não vir um captcha e o script executar completamente com sucesso, tente executar o script novamente até encontrar um captcha.

In [ ]:
%%writefile captcha_with_nova_act.py
from bedrock_agentcore.tools.browser_client import browser_session
from nova_act import NovaAct, BOOL_SCHEMA, ActAgentError
from rich.console import Console
from rich.panel import Panel
import sys
import json
import time
import argparse
sys.path.append("../interactive_tools")
from browser_viewer import BrowserViewerServer


console = Console()

from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print("usando região", region)

def contains_human_validation_error(err):
    """
    Verifica recursivamente se o erro ou seu atributo message indica HumanValidationError.
    """
    if err is None:
        return False

    # Verificação direta de string
    if isinstance(err, str) and "HumanValidationError" in err:
        return True

    # Se err tem atributo 'message' que é string ou outro erro, recursão
    if hasattr(err, "message"):
        return contains_human_validation_error(err.message)

    # Se err tem representação em string contendo o texto do erro
    if "HumanValidationError" in str(err):
        return True

    return False

def live_view_with_nova_act(steps, starting_page, nova_act_key, region="us-west-2"):
    """Executa o visualizador de navegador ao vivo com dimensionamento de tela."""
    console.print(
        Panel(
            "[bold cyan]Visualizador de Navegador ao Vivo[/bold cyan]\n\n"
            "Isso demonstra:\n"
            "• Visualização de navegador ao vivo com DCV\n"
            "• Tamanhos de tela configuráveis (não limitado a 900×800)\n"
            "• Callbacks de layout de tela adequados\n\n"
            "[yellow]Nota: Requer arquivos SDK do Amazon DCV[/yellow]",
            title="Visualizador de Navegador ao Vivo",
            border_style="blue",
        )
    )
    result = None

    try:
        # Passo 1: Criar sessão de navegador
        with browser_session(region) as client:
            ws_url, headers = client.generate_ws_headers()

            # Passo 2: Iniciar servidor visualizador
            console.print("\n[cyan]Passo 3: Iniciando servidor visualizador...[/cyan]")
            viewer = BrowserViewerServer(client, port=8000)
            viewer_url = viewer.start(open_browser=True)

            # Passo 3: Mostrar recursos
            console.print("\n[bold green]Recursos do Visualizador:[/bold green]")
            console.print(
                "• Tela padrão: 1600×900 (configurado via callback displayLayout)"
            )
            console.print("• Opções de tamanho: 720p, 900p, 1080p, 1440p")
            console.print("• Atualizações de tela em tempo real")
            console.print("• Funcionalidade de assumir/liberar controle")

            console.print("\n[yellow]Pressione Ctrl+C para parar[/yellow]")

            # Passo 4: Usar Nova Act para interagir com o navegador
            with NovaAct(
                cdp_endpoint_url=ws_url,
                cdp_headers=headers,
                preview={"playwright_actuation": True},
                nova_act_api_key=nova_act_key,
                starting_page=starting_page,
            ) as nova_act:
                
                for step_index, step in enumerate(steps):
                    max_retries = 3
                    retry_count = 0
                    
                    while retry_count < max_retries:
                        try:
                            print(f"Executando passo {step_index + 1}/{len(steps)}: {step}")
                            result = nova_act.act(step)
                            console.print(f"\n[bold green]Resultado Passo {step_index + 1}:[/bold green] {result}")
                            break  # Sucesso, mover para o próximo passo
                            
                        except ActAgentError as err:
                            # Verificar validação humana na mensagem ou estrutura do erro
                            if contains_human_validation_error(err):
                                print("CAPTCHA detectado! Por favor, resolva-o no navegador.")
                                captcha_wait_attempts = 0
                                max_captcha_wait_attempts = 8
                                
                                while captcha_wait_attempts < max_captcha_wait_attempts:
                                    try:
                                        time.sleep(10)  # Dar tempo ao usuário para resolver o captcha
                                        captcha_result = nova_act.act(
                                            "Há um captcha na tela?", schema=BOOL_SCHEMA
                                        )
                                        
                                        if captcha_result.matches_schema and not captcha_result.parsed_response:
                                            print("Captcha resolvido, continuando com o passo atual...")
                                            # Não incrementar retry_count para retentar o passo atual sem penalidade
                                            break
                                        else:
                                            print(f"Captcha ainda presente. Aguardando... (Tentativa {captcha_wait_attempts + 1}/{max_captcha_wait_attempts})")
                                            captcha_wait_attempts += 1
                                            
                                    except Exception as captcha_check_err:
                                        print(f"Erro ao verificar status do captcha: {str(captcha_check_err)}")
                                        captcha_wait_attempts += 1
                                        time.sleep(5)
                                
                                if captcha_wait_attempts >= max_captcha_wait_attempts:
                                    print("Número máximo de tentativas de espera do captcha atingido. Tentando continuar mesmo assim.")
                                    retry_count += 1
                                
                            else:
                                print(f"Erro não relacionado a captcha ocorreu: {str(err)}")
                                retry_count += 1
                                time.sleep(5)
                                
                        except Exception as general_err:
                            print(f"Erro inesperado no passo {step_index + 1}: {str(general_err)}")
                            retry_count += 1
                            time.sleep(5)
                            
                    if retry_count >= max_retries:
                        console.print(f"\n[bold red]Falha ao completar passo {step_index + 1} após {max_retries} tentativas.[/bold red]")
                        if step_index < len(steps) - 1:
                            console.print("[yellow]Tentando continuar com o próximo passo...[/yellow]")
                
                # Resumo final
                console.print("\n[bold blue]Execução da Tarefa Completa[/bold blue]")
        
    except Exception as e:
        console.print(f"\n[red]Erro: {e}[/red]")
        import traceback
        traceback.print_exc()
    finally:
        console.print("\n\n[yellow]Encerrando...[/yellow]")
        if "client" in locals():
            client.stop()
            console.print("✅ Sessão de navegador terminada")
    return result


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--steps", required=False, help="Array JSON ou lista separada por vírgulas de passos a executar", 
                        default='["Procure por notícias de IA e pressione enter. Se já houver notícias de IA digitadas na barra de busca, não faça nada", "Obtenha o primeiro resultado de notícias de IA, abra a página e extraia o título. Em vez disso, se você vir um resumo de IA, extraia o primeiro parágrafo do resumo e retorne"]')
    parser.add_argument("--starting-page", required=True, help="URL inicial")
    parser.add_argument("--nova-act-key", required=True, help="Chave de API Nova Act")
    parser.add_argument("--region", default="us-west-2", help="Região AWS")
    args = parser.parse_args()

    # Analisar passos - aceitar array JSON ou valores separados por vírgula
    try:
        # Tentar analisar como JSON primeiro
        steps = json.loads(args.steps)
    except json.JSONDecodeError:
        # Se não for JSON válido, tratar como string separada por vírgulas
        steps = [step.strip() for step in args.steps.split(',')]

    # Garantir que steps é uma lista
    if not isinstance(steps, list):
        steps = [steps]

    result = live_view_with_nova_act(
        steps, args.starting_page, args.nova_act_key, args.region
    )

In [ ]:
!python captcha_with_nova_act.py  --starting-page "https://www.google.com/" --nova-act-key {NOVA_ACT_KEY}

# Parabéns!